# §13 — Model Experiments & MLflow Comparison

Compares baseline and tuned XGBoost runs, learning curves, hyperparameter search distributions, and MLflow Model Registry versions.

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')
# move from notebooks/ to project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient

TRACKING_URI = 'sqlite:///experiments/mlflow/mlflow.db'
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)
EXPERIMENT_NAME = 'lead_bank_ranking'
print('MLflow tracking URI:', TRACKING_URI)


## 1. Experiments & Run Overview

In [ ]:
experiments = mlflow.search_experiments()
for exp in experiments:
    print(f'  {exp.name}  (id={exp.experiment_id})')


In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=500,
)
print(f'Total runs: {len(runs_df)}')
show_cols = [c for c in [
    'run_id', 'tags.mlflow.runName', 'tags.run_type',
    'start_time', 'status',
    'metrics.val_auc', 'metrics.val_ndcg_3', 'metrics.val_mrr',
] if c in runs_df.columns]
runs_df[show_cols].sort_values('start_time', ascending=False).head(30)


## 2. Baseline vs Tuned — Final Metrics

In [ ]:
with open('models/v1/metadata.json') as f:
    meta = json.load(f)

METRICS = ['auc_roc', 'ndcg_at_1', 'ndcg_at_3', 'ndcg_at_5',
           'recall_at_3', 'mrr', 'f1_class1']
THRESHOLDS = {'auc_roc': 0.82, 'ndcg_at_3': 0.70,
              'recall_at_3': 0.75, 'mrr': 0.60, 'f1_class1': 0.65}

# Tuned final model metrics from metadata
tuned_val  = meta.get('val_metrics', {})
tuned_test = meta.get('test_metrics', {})

rows = []
rows.append({'Split': 'Val',  'Model': 'Tuned v2 @staging',
             **{m: tuned_val.get(m, float('nan')) for m in METRICS}})
rows.append({'Split': 'Test', 'Model': 'Tuned v2 @staging',
             **{m: tuned_test.get(m, float('nan')) for m in METRICS}})

cmp = pd.DataFrame(rows).set_index(['Split', 'Model'])
print('=== Final Model Metrics ===')
print(cmp.round(4).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
test_vals = [tuned_test.get(m, 0) for m in METRICS]
bars = ax.bar(METRICS, test_vals, color='#5b9bd5', edgecolor='white', alpha=0.85)
# Threshold lines
for i, m in enumerate(METRICS):
    if m in THRESHOLDS:
        ax.hlines(THRESHOLDS[m], i - 0.4, i + 0.4, colors='red',
                  linestyles='--', linewidth=1.5)
ax.set_ylabel('Score')
ax.set_title('Tuned v2 — Test Metrics (red dashes = §11 thresholds)')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
# Annotate bars
for bar, val in zip(bars, test_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('notebooks/model_comparison.png', dpi=150)
plt.show()


## 3. Optuna Trial History

In [ ]:
# Nested trial runs have a parentRunId tag
trial_runs = pd.DataFrame()
if 'tags.mlflow.parentRunId' in runs_df.columns:
    trial_runs = runs_df[runs_df['tags.mlflow.parentRunId'].notna()].copy()
print(f'Trial runs found: {len(trial_runs)}')

if not trial_runs.empty and 'metrics.val_ndcg_3' in trial_runs.columns:
    best_idx = trial_runs['metrics.val_ndcg_3'].idxmax()
    best = trial_runs.loc[best_idx]
    print(f'Best trial NDCG@3: {best["metrics.val_ndcg_3"]:.4f}')
    p_cols = [c for c in trial_runs.columns if c.startswith('params.')]
    if p_cols:
        print('Best params:', best[p_cols].to_dict())


In [ ]:
if not trial_runs.empty and 'metrics.val_ndcg_3' in trial_runs.columns:
    scores = trial_runs['metrics.val_ndcg_3'].dropna().reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].scatter(range(len(scores)), scores.values,
                   color='#5b9bd5', alpha=0.7, s=30, zorder=2)
    axes[0].axhline(scores.max(), color='red', linestyle='--',
                   label=f'Best = {scores.max():.4f}')
    axes[0].set_xlabel('Trial index')
    axes[0].set_ylabel('Val NDCG@3')
    axes[0].set_title('Optuna Trial Scores')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    cum_best = scores.cummax()
    axes[1].plot(range(len(cum_best)), cum_best.values,
                color='#ed7d31', linewidth=2)
    axes[1].set_xlabel('Trial index')
    axes[1].set_ylabel('Best Val NDCG@3 so far')
    axes[1].set_title('Convergence Plot')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('notebooks/optuna_convergence.png', dpi=150)
    plt.show()
else:
    print('Trial metrics not found — run the tuner to populate.')


## 4. Learning Curves — Tuned Model

In [ ]:
TUNED_RUN_ID = meta['run_id']
print(f'Loading learning curves for run: {TUNED_RUN_ID}')

def get_history(run_id, metric_key):
    try:
        hist = client.get_metric_history(run_id, metric_key)
        return [(h.step, h.value) for h in sorted(hist, key=lambda x: x.step)]
    except Exception:
        return []

tr_auc = get_history(TUNED_RUN_ID, 'train_iter_auc')
vl_auc = get_history(TUNED_RUN_ID, 'val_iter_auc')
tr_ll  = get_history(TUNED_RUN_ID, 'train_iter_logloss')
vl_ll  = get_history(TUNED_RUN_ID, 'val_iter_logloss')

print(f'  train_iter_auc steps : {len(tr_auc)}')
print(f'  val_iter_auc   steps : {len(vl_auc)}')
print(f'  train_iter_logloss   : {len(tr_ll)}')
print(f'  val_iter_logloss     : {len(vl_ll)}')


In [ ]:
if tr_auc and vl_auc:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    tr_s, tr_v = zip(*tr_auc)
    vl_s, vl_v = zip(*vl_auc)
    axes[0].plot(tr_s, tr_v, label='Train AUC', color='#5b9bd5', linewidth=1.5)
    axes[0].plot(vl_s, vl_v, label='Val AUC',   color='#ed7d31', linewidth=1.5)
    axes[0].set_xlabel('Boosting Iteration')
    axes[0].set_ylabel('AUC')
    axes[0].set_title('Learning Curve — AUC')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    if tr_ll and vl_ll:
        tls, tlv = zip(*tr_ll)
        vls, vlv = zip(*vl_ll)
        axes[1].plot(tls, tlv, label='Train LogLoss', color='#5b9bd5', linewidth=1.5)
        axes[1].plot(vls, vlv, label='Val LogLoss',   color='#ed7d31', linewidth=1.5)
        axes[1].set_xlabel('Boosting Iteration')
        axes[1].set_ylabel('Log-Loss')
        axes[1].set_title('Learning Curve — Log-Loss')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('notebooks/learning_curves.png', dpi=150)
    plt.show()
else:
    print('Per-iteration metrics not available for this run.')


## 5. Best Hyperparameters

In [ ]:
with open('models/v1/best_hyperparams.json') as f:
    best_hp = json.load(f)

print(f'Best trial NDCG@3: {best_hp["best_cv_ndcg3"]:.4f}')
print()
for k, v in best_hp.get('best_params', {}).items():
    print(f'  {k:<30} {v:.6f}' if isinstance(v, float) else f'  {k:<30} {v}')


## 6. MLflow Model Registry

In [ ]:
REGISTRY_NAME = 'lead_bank_xgb_ranker'
try:
    versions = client.search_model_versions(f"name='{REGISTRY_NAME}'")
    print(f'Model: {REGISTRY_NAME}  ({len(versions)} versions)')
    for v in sorted(versions, key=lambda x: int(x.version)):
        aliases = getattr(v, 'aliases', [])
        print(f'  v{v.version}  stage={v.current_stage}  aliases={aliases}  '
              f'run={v.run_id[:12]}...')
except Exception as e:
    print(f'Registry error: {e}')


## 7. Feature Importance (Tuned Model)

In [ ]:
import pickle
from xgboost import XGBClassifier

with open('models/v1/preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

model = XGBClassifier()
model.load_model('models/v1/xgb_model.ubj')

importances = model.get_booster().get_score(importance_type='gain')
imp_df = pd.DataFrame(list(importances.items()), columns=['feature', 'importance'])
imp_df = imp_df.sort_values('importance', ascending=False)

top20 = imp_df.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
y = range(len(top20))
ax.barh(list(y), top20['importance'].values[::-1], color='steelblue', edgecolor='white')
ax.set_yticks(list(y))
ax.set_yticklabels(top20['feature'].values[::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('Top-20 Feature Importances — Tuned XGBoost')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/feature_importance_nb.png', dpi=150)
plt.show()

required = {'cibil_gap', 'foir_headroom', 'bureau_fatigue_flag',
            'income_type_match', 'amount_fit_flag'}
top10 = set(imp_df.head(10)['feature'].tolist())
print('\n§14 requirement: must appear in top-10 features')
for feat in sorted(required):
    status = 'PASS' if feat in top10 else 'FAIL'
    print(f'  [{status}] {feat}')
